In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Clear filesystem cache (pagecache, dentries, inodes)
!sudo sync
!sudo sh -c 'echo 3 > /proc/sys/vm/drop_caches'

sh: 1: cannot create /proc/sys/vm/drop_caches: Read-only file system


In [3]:
# Check GPU status
!nvidia-smi

Sat Mar 28 19:12:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.56.2
%pip install --no-deps trl==0.22.2
%pip install faiss-gpu-cu12

# Configuration

In [5]:
# Fix TorchDynamo bug
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

In [6]:
# Model configuration
MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'

# Data configuration
SFT_DATA_ID = 'jxie/coco_captions'
DATA_SPLIT = 'train'
DATA_SIZE = 1250
TEST_RATIO = 0.1

BVIR_DATA_ID = 'alxxtexxr/BIVR-Data'
CODEBOOK_PATH = 'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'

# Data

In [7]:
from datasets import load_dataset, Dataset

def load_hf_dataset(data_id, split, total_size):
    dataset_stream = load_dataset(data_id, split=split, streaming=True)
    
    unique_cocoids = set()
    sliced_data = []
    for example in dataset_stream:
        if len(sliced_data) >= total_size:
            break

        cocoid = example.get('cocoid', None)
        if cocoid is not None:
            if cocoid in unique_cocoids:
                continue
            unique_cocoids.add(cocoid)

        sliced_data.append(example)

    dataset = Dataset.from_list(sliced_data)
    return dataset

# Load all split sets
train_set = load_hf_dataset(SFT_DATA_ID, split='train', total_size=int(DATA_SIZE*(1-TEST_RATIO*2)))
val_set = load_hf_dataset(SFT_DATA_ID, split='validation', total_size=int(DATA_SIZE*TEST_RATIO))
test_set = load_hf_dataset(SFT_DATA_ID, split='test', total_size=int(DATA_SIZE*TEST_RATIO))

print("Train dataset:")
print(train_set)
print("\nValidation dataset:")
print(val_set)
print("\nTest dataset:")
print(test_set)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

Train dataset:
Dataset({
    features: ['image', 'filename', 'cocoid', 'caption'],
    num_rows: 1000
})

Validation dataset:
Dataset({
    features: ['image', 'filename', 'cocoid', 'caption'],
    num_rows: 125
})

Test dataset:
Dataset({
    features: ['image', 'filename', 'cocoid', 'caption'],
    num_rows: 125
})


In [ ]:
from datasets import concatenate_datasets

# Add a 'source' column to each split set
train_set = train_set.add_column('source', ['train'] * len(train_set))
val_set   = val_set.add_column('source', ['val'] * len(val_set))
test_set  = test_set.add_column('source', ['test'] * len(test_set))

# Combine all split sets into one dataset
dataset = concatenate_datasets([train_set, val_set, test_set])
print(dataset)

Dataset({
    features: ['image', 'filename', 'cocoid', 'caption', 'source'],
    num_rows: 1250
})


In [9]:
from PIL import Image

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

# # Resize to (512, 512) and convert to RGB
# train_set = train_set.map(process_image)
# val_set = val_set.map(process_image)
# test_set = test_set.map(process_image)

# # Remove the original 'image' column
# train_set = train_set.remove_columns('image')
# val_set = val_set.remove_columns('image')
# test_set = test_set.remove_columns('image')

# # Rename the 'decoded_image' column to 'image'
# train_set = train_set.rename_column('decoded_image', 'image')
# val_set = val_set.rename_column('decoded_image', 'image')
# test_set = test_set.rename_column('decoded_image', 'image')

# print("Train dataset:")
# print(train_set)
# print("\nValidation dataset:")
# print(val_set)
# print("\nTest dataset:")
# print(test_set)

dataset = dataset.map(process_image) # Resize to (512, 512) and convert to RGB
dataset = dataset.remove_columns('image') # Remove the original 'image' column
dataset = dataset.rename_column('decoded_image', 'image') # Rename the 'decoded_image' column to 'image'
print(dataset)

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

Dataset({
    features: ['filename', 'cocoid', 'caption', 'source', 'image'],
    num_rows: 1250
})


# Model

In [10]:
from unsloth import FastVisionModel
from transformers import AutoProcessor

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = False, # False for LoRA 16bit
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # max_seq_length = 16384, # Must be this long for VLMs
    # fast_inference = True, # Enable vLLM fast inference
    # fast_inference = False, # Disable to fix the vLLM bug on T4
    # gpu_memory_utilization = 0.8, # Reduce if out of memory
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# Visual Codebook

In [11]:
from huggingface_hub import hf_hub_download

codebook_path = hf_hub_download(
    repo_id=BVIR_DATA_ID,
    filename=CODEBOOK_PATH,
    repo_type='dataset',
)
print("Codebook downloaded to:", codebook_path)

Codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--BIVR-Data/snapshots/5de7da8ec1b2cd2138e50bc11b18e17d76cbf942/model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy


In [12]:
import numpy as np
import faiss

codebook = np.load(codebook_path).astype(np.float32) # (K, 3584) -> Current: (8192, 3584)
faiss.normalize_L2(codebook)

print("Codebook shape:", codebook.shape)
print("Mean L2 norm (should be ~1):", np.linalg.norm(codebook, axis=1).mean())

Codebook shape: (8192, 3584)
Mean L2 norm (should be ~1): 1.0


In [13]:
d = codebook.shape[1] # Current: 3584

# FAISS search (L2 is equivalent to cosine for unit vectors)
index = faiss.IndexFlatL2(d)
index.add(codebook)

# Image Serialization

In [15]:
distances_all = []

def serialize_img(example):
    image = example['image']
    inputs = processor.image_processor(images=[image], return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(model.device)     # Current: (1296, 1176)
    image_grid_thw = inputs['image_grid_thw'].to(model.device) # Current: (1, 3)
    
    # Get visual embeddings of the image from the model
    with torch.no_grad():
        visual_embs = model.visual(pixel_values, image_grid_thw) # Current: (324, 3584)
    visual_embs = visual_embs.cpu().float().numpy() # Convert to Float32 NumPy array
    faiss.normalize_L2(visual_embs)                 # Normalize to unit length
    
    # Perform FAISS search to find nearest codebook vector for each visual embedding
    distances, codebook_idxs = index.search(visual_embs, 1)
    distances, codebook_idxs = distances.flatten(), codebook_idxs.flatten()
    # distances     -> Current: (324,)
    # codebook_idxs -> Current: (324,)
    distances_all.extend(distances)
    
    # Serialize image into a string format using the codebook indices (e.g., "<|vtok_123|><|vtok_456|>...")
    image_serialized = "".join([f'<|vtok_{i}|>' for i in codebook_idxs])
    example['image_serialized'] = image_serialized
    return example

dataset = dataset.map(serialize_img)

Parameter 'function'=<function serialize_img at 0x7a4b61a1d3a0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

In [18]:
from datasets import DatasetDict

# Split the dataset back into train, val, test sets based on the 'source' column
train_set = dataset.filter(lambda x: [s == 'train' for s in x['source']], batched=True)
val_set   = dataset.filter(lambda x: [s == 'val' for s in x['source']], batched=True)
test_set  = dataset.filter(lambda x: [s == 'test' for s in x['source']], batched=True)

# Create a DatasetDict for the split sets
dataset_split = DatasetDict({
    'train': train_set,
    'val': val_set,
    'test': test_set,
})
print(dataset_split)

Filter:   0%|          | 0/1250 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1250 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['filename', 'cocoid', 'caption', 'source', 'image', 'image_serialized'],
        num_rows: 1000
    })
    val: Dataset({
        features: ['filename', 'cocoid', 'caption', 'source', 'image', 'image_serialized'],
        num_rows: 125
    })
    test: Dataset({
        features: ['filename', 'cocoid', 'caption', 'source', 'image', 'image_serialized'],
        num_rows: 125
    })
})


In [26]:
# Compute RMSE
rmse = np.sqrt(np.vstack(distances_all).mean())
print("RMSE:", rmse)

RMSE: 0.7064397


In [ ]:
# Push the final dataset to Hugging Face Hub
dataset_split.push_to_hub(f"alxxtexxr/{SFT_DATA_ID.split('/')[-1]}_{DATA_SIZE/1000}k_serialized")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   5%|5         | 2.62MB / 49.8MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/coco_captions_1.25k/commit/13e9a2f6a0e6f03c5aef0bff3b18d2d5e8820c5f', commit_message='Upload dataset', commit_description='', oid='13e9a2f6a0e6f03c5aef0bff3b18d2d5e8820c5f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/coco_captions_1.25k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/coco_captions_1.25k'), pr_revision=None, pr_num=None)